### Make sure the datasets are SHUFFLED

In [1]:
import glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataset_utils import *

pi = 3.14159265359
maxval=1e9
minval=1e-9

In [2]:
from dataloaders.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from models.mlp_encoder_model_nonquantized import *
#from models.conv2d_model_nonquantized import *

2025-10-07 19:53:52.698869: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-07 19:53:53.443146: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
def get_thresholds(model):
    """Map soft quantization thresholds & levels from normalized to raw charge space"""
    
    # Extract soft quantization parameters
    sq_layer = model.get_layer(name="soft_quantizer_output")
    thresholds = sq_layer.thresholds.numpy()

    print("QUANTIZATION THRESHOLDS (e):")
    print(f"  Thresholds: {thresholds}") 
    
    return thresholds

In [4]:
def save_quantized_manual_parquet(input_filepath, output_dir, charge_thresholds, quant_values, shuffled=True, noise=-1, min_threshold=None, max_threshold=None):
    """
    charge_levels = list of 3 values indicating the charge boundaries for the 2-bit binnings
    quant_values = list of 4 values representing the output value that the bins will be mapped to
    noise = add gaussian noise in the form of [mu, sigma] (if -1 then no noise will be added)
    threshold = zero all charges <= threshold value (if -1 then no threshold will be added)
    shuffled = boolean representing if the dataset is shuffled or unshuffled
    """
    temp_df = pd.read_parquet(input_filepath)
    if noise != -1:
        temp_df = add_noise(x=temp_df, mu=noise[0], sig=noise[1], shuffled=shuffled, seed=None)
    if min_threshold != None:
        temp_df = apply_threshold(x=temp_df, thresh=threshold, minimum=True, shuffled=shuffled)
    if max_threshold != None:
        temp_df = apply_threshold(x=temp_df, thresh=threshold, minimum=False, shuffled=shuffled)
    temp_quantized_df = quantize_manual(x=temp_df, charge_levels=charge_thresholds, quant_values=quant_values, shuffled=shuffled)
    output_filepath = output_dir + input_filepath.split('/')[-1]
    temp_quantized_df.to_parquet(output_filepath)
    print(f'{output_filepath} successfully processed and saved.')

In [6]:
#model=CreateModel_Full_SoftQuantizer((16,16,2), initial_thresholds=[400, 1000, 2000], threshold_offset=0.0)
model=CreateModel_Slim_SoftQuantizer((16,16,2), initial_thresholds=[400, 1000, 2000], threshold_offset=0.0)
#model=CreateModel_Full_SoftQuantizer((16,16,2), n_filters=5, pool_size=3, initial_thresholds=[400, 1000, 2000], threshold_offset=0.0)

model.summary()

# get best weights file
pitch = '50x12P5'
batch_size = 5000
fingerprint = '34c2da80'
timeslices = 2
files = os.listdir('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-soft_quantizer-checkpoints'.format(pitch, batch_size, fingerprint, timeslices))

vlosses = [float(f.split("-v")[1].split(".hdf5")[0]) for f in files]
bestfile = files[np.argmin(vlosses)]
model.load_weights('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-soft_quantizer-checkpoints/'.format(pitch, batch_size, fingerprint, timeslices)+bestfile)

print('Best model: {}'.format(bestfile))

Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls (InputLayer)     [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 soft_quantizer_output (Sof  (None, 16, 16, 2)            8         ['input_pxls[0][0]']          
 tQuantizeLayer)                                                                                  
                                                                                                  
 average_pooling2d_2 (Avera  (None, 16, 1, 2)             0         ['soft_quantizer_output[0][0]'
 gePooling2D)                                                       ]                             
                                                                                 

In [7]:
# load in the test set
test_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = '/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/TFR_files/{}t/TFR_val_contained_slim'.format(timeslices),
    quantize = False # False for soft quantizer and manually quantized inputs
)

Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/TFR_files/2t/TFR_val_contained_slim/metadata.json


In [8]:
# Soft Quantize Layer
layer = model.get_layer("soft_quantizer_output")  # or model.layers[0]
print('---Soft Quantize Layer Details---')
levels = layer.levels.numpy()
charge_thresholds = get_thresholds(model)
print(f'  Levels: {levels}')

---Soft Quantize Layer Details---
QUANTIZATION THRESHOLDS (e):
  Thresholds: [ 381.9998   972.99963 1980.0027 ]
  Levels: [-1.         -0.33333337  0.33333325  0.9999999 ]


##### Create new output training and validation/test directories to save the datasets that will be manually digitized according to the optimized charge thresholds

In [8]:
output_train_dir = '/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/train_contained_digitize-manual_conv2d-FULL/'
output_test_dir = '/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/test_contained_digitize-manual_conv2d-FULL/'

dirs_to_create = [
    output_train_dir,
    output_test_dir
]

# Create each directory if it doesn't exist
for directory in dirs_to_create:
    os.makedirs(directory, exist_ok=True)

In [9]:
train_files = glob.glob('/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/train_contained/*.parquet')
test_files = glob.glob('/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/test_contained/*.parquet')

In [10]:
for file in train_files:
    save_quantized_manual_parquet(input_filepath=file, output_dir=output_train_dir, charge_thresholds=charge_thresholds, quant_values=levels)

/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/train_contained_digitize-manual_mlp-FULL/part.41.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/train_contained_digitize-manual_mlp-FULL/part.26.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/train_contained_digitize-manual_mlp-FULL/part.74.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/train_contained_digitize-manual_mlp-FULL/part.1.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/train_contained_digitize-manual_mlp-FULL/part.49.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50

In [11]:
for file in test_files:
    save_quantized_manual_parquet(input_filepath=file, output_dir=output_test_dir, charge_thresholds=charge_thresholds, quant_values=levels)

/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/test_contained_digitize-manual_mlp-FULL/part.83.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/test_contained_digitize-manual_mlp-FULL/part.80.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/test_contained_digitize-manual_mlp-FULL/part.95.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/test_contained_digitize-manual_mlp-FULL/part.92.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/test_contained_digitize-manual_mlp-FULL/part.93.parquet successfully processed and saved.
/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P